## Simple Mission in Guided-Mode

This notebook serves to get familiar with the simulator

In [ ]:
import os
import time

from simulator.config import (
    ARDUPILOT_VEHICLE_PATH,
    DATA_PATH,
    LOGS_PATH,
    BasePort,
)
from simulator.helpers import (
    DataLogger,
    clean,
    create_process,
    setup_logging,
    terminate_process_group,
)
from simulator.helpers.connections import create_tcp_conn, wait_for_port
from simulator.helpers.coordinates import GRAPose
from simulator.planner import GuidedPlan, State
from simulator.runtime import MAVLinkManager

clean()

## Create Plan in guided mode

Action commands are sent online, in contrast to auto mode where the full plan is loaded into the UAV before starting the mission.

In [ ]:
gra_origin = GRAPose(lat=-35.3633245,lon=149.1652241,alt=0.0,heading=0)

sysid = 1
speedup = 19
plan = GuidedPlan.square_traj(side_len=5, alt=5)
plan

## Launch Copter (ardupilot)

In [ ]:
# #This must agree with first waypoint in mission.waypoint
spawn_str= gra_origin.to_str()
ARDUCOPTER_BIN = os.path.expanduser("~/ardupilot/build/sitl/bin/arducopter")
autotest_dir = os.path.dirname(os.path.expanduser(ARDUPILOT_VEHICLE_PATH))


vehicle_cmd = [
    ARDUCOPTER_BIN,
    "--model", "quad",
    "--speedup", str(speedup),
    "--sysid", str(sysid),
    "--base-port", str(BasePort.ARP), 
    "--sim-address", "127.0.0.1",
    "--home", spawn_str,
    "--defaults", os.path.join(autotest_dir, "default_params/copter.parm"),
]

veh_proc = create_process(
    " ".join(vehicle_cmd),   # ✅ THIS is the fix
    after="",
    visible=False,
    suppress_output=True,
    title="ardu_vehicle",
    new_process_group=True,
)
wait_for_port(BasePort.ARP, timeout=1,verbose=True)

## Connect to the vehicle

In [ ]:
conn = create_tcp_conn(
    base_port=BasePort.ARP, 
    offset=0,
    role="client", 
    src_sysid=sysid,
    src_compid=140,
    
)

print("✅ TCP connection established!")

## Start mavlink manager

In [ ]:
data_logger = DataLogger(path=DATA_PATH / "msgs", sysid=sysid)
mav_mng = MAVLinkManager(
    conn=conn,
    data_logger=data_logger
)
mav_mng.start()

# Execute Plan in guided mode

In [ ]:
setup_logging(LOGS_PATH/'plan',verbose=3)
plan.bind(gra_origin.unpose(), mav_mng)
while plan.state != State.DONE:
    plan.act()
    time.sleep(0.1)

## Close connection and vehicle process

In [ ]:
mav_mng.stop()
terminate_process_group(veh_proc,"vehicle_process")

In [ ]:
print(plan)

In [ ]:
plan.reset()
print(plan)